                 THE HIGHWAY.MP4
                        │
                        ▼
                ┌───────────────┐
                │    OpenCV     │
                │ Video Reader  │
                └───────┬───────┘
                        │
                        ▼
                ┌───────────────┐
                │     YOLO      │
                │Object Detection│
                └───────┬───────┘
                        │
              Bounding Boxes
                        │
                        ▼
                ┌───────────────┐
                │   ByteTrack   │
                │    Tracking   │
                └───────┬───────┘
                        │
                     Track ID
                        │
             ┌──────────┴──────────┐
             ▼                     ▼
       Annotated Video        Detection Data
                                   │
                                   ▼
                              Pandas DataFrame
                                   │
                 ┌─────────────────┼─────────────────┐
                 ▼                 ▼                 ▼
              CSV file        Trajectories      Analytics

In [2]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.0 MB/s eta 0:00:00


In [3]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [4]:
VIDEO_PATH = "The Highway.mp4"

OUTPUT_VIDEO = "traffic_detection_tracking.mp4"
OUTPUT_CSV = "traffic_detections.csv"

CAMERA_ID = "CAM_01"

# Optional camera information
CAMERA_LATITUDE = 28.6139
CAMERA_LONGITUDE = 77.2090

print("Video:", VIDEO_PATH)
print("Camera:", CAMERA_ID)

Video: The Highway.mp4
Camera: CAM_01


In [5]:
model = YOLO("yolo11n.pt")

print("YOLO model loaded")

YOLO model loaded


In [7]:
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

duration = total_frames / fps if fps > 0 else 0

print("FPS:", fps)
print("Width:", width)
print("Height:", height)
print("Total frames:", total_frames)
print("Duration:", round(duration, 2), "seconds")

FPS: 30.0
Width: 256
Height: 144
Total frames: 180
Duration: 6.0 seconds


In [8]:
TRAFFIC_CLASSES = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

print(TRAFFIC_CLASSES)

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}


In [9]:
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():
    raise RuntimeError("Could not create output video")

print("Output video created")

Output video created


In [10]:
records = []

frame_number = 0

print("Ready for detection and tracking")

Ready for detection and tracking


In [11]:
cap = cv2.VideoCapture(VIDEO_PATH)

frame_number = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    timestamp = frame_number / fps

    # YOLO detection + ByteTrack tracking
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.30,
        verbose=False
    )

    result = results[0]

    # Annotated frame
    annotated_frame = frame.copy()

    if result.boxes is not None:

        boxes = result.boxes

        xyxy = boxes.xyxy.cpu().numpy()
        confidences = boxes.conf.cpu().numpy()
        class_ids = boxes.cls.cpu().numpy().astype(int)

        # Track IDs
        if boxes.id is not None:
            track_ids = boxes.id.cpu().numpy().astype(int)
        else:
            track_ids = [-1] * len(xyxy)

        for box, confidence, class_id, track_id in zip(
            xyxy,
            confidences,
            class_ids,
            track_ids
        ):

            # Only traffic-related classes
            if class_id not in TRAFFIC_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box)

            object_name = TRAFFIC_CLASSES[class_id]

            # Center point
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            # Store record
            records.append({
                "frame": frame_number,
                "timestamp_sec": round(timestamp, 3),
                "track_id": int(track_id),
                "class_id": int(class_id),
                "class_name": object_name,
                "confidence": round(float(confidence), 4),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "center_x": center_x,
                "center_y": center_y,
                "camera_id": CAMERA_ID,
                "camera_latitude": CAMERA_LATITUDE,
                "camera_longitude": CAMERA_LONGITUDE
            })

            # Draw bounding box
            cv2.rectangle(
                annotated_frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # Label
            label = f"{object_name} ID:{track_id} {confidence:.2f}"

            cv2.putText(
                annotated_frame,
                label,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

            # Center point
            cv2.circle(
                annotated_frame,
                (center_x, center_y),
                4,
                (0, 0, 255),
                -1
            )

    # Frame information
    cv2.putText(
        annotated_frame,
        f"Frame: {frame_number}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.putText(
        annotated_frame,
        f"Time: {timestamp:.2f}s",
        (20, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.putText(
        annotated_frame,
        f"Camera: {CAMERA_ID}",
        (20, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    # Write frame
    out.write(annotated_frame)

    # Progress
    if frame_number % 100 == 0:
        print(
            f"Processed {frame_number}/{total_frames} frames"
        )

cap.release()
out.release()

print("Processing completed.")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 434ms
Prepared 1 package in 178ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processed 100/180 frames
Processing completed.


In [12]:
df = pd.DataFrame(records)

print("Number of detections:", len(df))

df.head()

Number of detections: 2219


,frame,timestamp_sec,track_id,class_id,class_name,confidence,x1,y1,x2,y2,center_x,center_y,camera_id,camera_latitude,camera_longitude
0,1,0.033,1,2,car,0.8427,50,90,95,132,72,111,CAM_01,28.6139,77.209
1,1,0.033,2,2,car,0.7934,76,86,109,119,92,102,CAM_01,28.6139,77.209
2,1,0.033,3,2,car,0.7368,146,78,185,115,165,96,CAM_01,28.6139,77.209
3,1,0.033,4,2,car,0.7056,33,130,78,143,55,136,CAM_01,28.6139,77.209
4,1,0.033,5,2,car,0.6839,85,67,116,94,100,80,CAM_01,28.6139,77.209


In [13]:
df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(f"CSV saved: {OUTPUT_CSV}")

CSV saved: traffic_detections.csv


In [14]:
df = pd.read_csv(OUTPUT_CSV)

print(df.shape)

df.head(20)

(2219, 15)


,frame,timestamp_sec,track_id,class_id,class_name,confidence,x1,y1,x2,y2,center_x,center_y,camera_id,camera_latitude,camera_longitude
0,1,0.033,1,2,car,0.8427,50,90,95,132,72,111,CAM_01,28.6139,77.209
1,1,0.033,2,2,car,0.7934,76,86,109,119,92,102,CAM_01,28.6139,77.209
2,1,0.033,3,2,car,0.7368,146,78,185,115,165,96,CAM_01,28.6139,77.209
3,1,0.033,4,2,car,0.7056,33,130,78,143,55,136,CAM_01,28.6139,77.209
4,1,0.033,5,2,car,0.6839,85,67,116,94,100,80,CAM_01,28.6139,77.209
5,1,0.033,6,2,car,0.6261,136,59,162,81,149,70,CAM_01,28.6139,77.209
6,1,0.033,7,5,bus,0.4666,139,29,161,54,150,41,CAM_01,28.6139,77.209
7,1,0.033,8,7,truck,0.4520,149,113,220,143,184,128,CAM_01,28.6139,77.209
8,1,0.033,9,2,car,0.4480,0,45,24,65,12,55,CAM_01,28.6139,77.209
9,1,0.033,10,7,truck,0.3947,160,34,175,51,167,42,CAM_01,28.6139,77.209


In [15]:
summary = (
    df.groupby("class_name")
      .agg(
          detections=("track_id", "count"),
          unique_objects=("track_id", "nunique"),
          average_confidence=("confidence", "mean")
      )
      .reset_index()
)

summary

,class_name,detections,unique_objects,average_confidence
0,bus,86,4,0.503659
1,car,1980,33,0.610660
2,truck,153,11,0.422688


In [16]:
unique_vehicle_count = df["track_id"].nunique()

print(
    "Unique tracked objects:",
    unique_vehicle_count
)

Unique tracked objects: 42


In [17]:
vehicle_counts = (
    df.groupby("class_name")["track_id"]
      .nunique()
      .sort_values(ascending=False)
)

vehicle_counts

,track_id
class_name,
car,33
truck,11
bus,4


In [18]:
vehicle_id = 1

vehicle_1 = df[
    df["track_id"] == vehicle_id
].sort_values("frame")

vehicle_1.head(20)

,frame,timestamp_sec,track_id,class_id,class_name,confidence,x1,y1,x2,y2,center_x,center_y,camera_id,camera_latitude,camera_longitude
0,1,0.033,1,2,car,0.8427,50,90,95,132,72,111,CAM_01,28.6139,77.209
12,2,0.067,1,2,car,0.8384,50,91,95,132,72,111,CAM_01,28.6139,77.209
24,3,0.100,1,2,car,0.8433,50,91,95,132,72,111,CAM_01,28.6139,77.209
37,4,0.133,1,2,car,0.8520,50,92,95,133,72,112,CAM_01,28.6139,77.209
51,5,0.167,1,2,car,0.8352,49,92,95,134,72,113,CAM_01,28.6139,77.209
65,6,0.200,1,2,car,0.8464,49,93,95,135,72,114,CAM_01,28.6139,77.209
79,7,0.233,1,2,car,0.8119,49,93,94,135,71,114,CAM_01,28.6139,77.209
92,8,0.267,1,2,car,0.8156,48,94,94,136,71,115,CAM_01,28.6139,77.209
105,9,0.300,1,2,car,0.7938,48,94,94,136,71,115,CAM_01,28.6139,77.209
118,10,0.333,1,2,car,0.8117,48,95,94,137,71,116,CAM_01,28.6139,77.209


In [19]:
df = df.sort_values(
    ["track_id", "frame"]
).copy()

df["previous_x"] = (
    df.groupby("track_id")["center_x"]
      .shift(1)
)

df["previous_y"] = (
    df.groupby("track_id")["center_y"]
      .shift(1)
)

df["pixel_distance"] = np.sqrt(
    (df["center_x"] - df["previous_x"]) ** 2 +
    (df["center_y"] - df["previous_y"]) ** 2
)

df["pixel_distance"] = df["pixel_distance"].fillna(0)

df.head()

,frame,timestamp_sec,track_id,class_id,class_name,confidence,x1,y1,x2,y2,center_x,center_y,camera_id,camera_latitude,camera_longitude,previous_x,previous_y,pixel_distance
0,1,0.033,1,2,car,0.8427,50,90,95,132,72,111,CAM_01,28.6139,77.209,NaN,NaN,0.0
12,2,0.067,1,2,car,0.8384,50,91,95,132,72,111,CAM_01,28.6139,77.209,72.0,111.0,0.0
24,3,0.100,1,2,car,0.8433,50,91,95,132,72,111,CAM_01,28.6139,77.209,72.0,111.0,0.0
37,4,0.133,1,2,car,0.8520,50,92,95,133,72,112,CAM_01,28.6139,77.209,72.0,111.0,1.0
51,5,0.167,1,2,car,0.8352,49,92,95,134,72,113,CAM_01,28.6139,77.209,72.0,112.0,1.0


In [20]:
trajectory = (
    df.groupby(["track_id", "class_name"])
      .agg(
          first_frame=("frame", "min"),
          last_frame=("frame", "max"),
          first_x=("center_x", "first"),
          first_y=("center_y", "first"),
          last_x=("center_x", "last"),
          last_y=("center_y", "last")
      )
      .reset_index()
)

trajectory.head(20)

,track_id,class_name,first_frame,last_frame,first_x,first_y,last_x,last_y
0,1,car,1,80,72,111,59,140
1,2,car,1,94,92,102,77,140
2,3,car,1,127,165,96,188,139
3,4,car,1,11,55,136,51,141
4,5,car,1,175,100,80,69,139
5,6,car,1,180,149,70,159,112
6,7,bus,1,179,150,41,161,48
7,7,truck,39,180,153,42,161,48
8,8,truck,1,9,184,128,188,129
9,9,car,1,132,12,55,43,44


In [21]:
trajectory.to_csv(
    "vehicle_trajectories.csv",
    index=False
)

print("Trajectory CSV saved.")

Trajectory CSV saved.


In [22]:
print("=" * 50)
print("TRAFFIC VIDEO ANALYTICS COMPLETED")
print("=" * 50)

print("Input video:", VIDEO_PATH)
print("Output video:", OUTPUT_VIDEO)
print("Detection CSV:", OUTPUT_CSV)
print("Trajectory CSV:", "vehicle_trajectories.csv")

print()
print("Total detection records:", len(df))
print("Unique tracked objects:", df["track_id"].nunique())

print()
print("Objects by class:")
print(
    df.groupby("class_name")["track_id"]
      .nunique()
      .sort_values(ascending=False)
)

TRAFFIC VIDEO ANALYTICS COMPLETED
Input video: The Highway.mp4
Output video: traffic_detection_tracking.mp4
Detection CSV: traffic_detections.csv
Trajectory CSV: vehicle_trajectories.csv

Total detection records: 2219
Unique tracked objects: 42

Objects by class:
class_name
car      33
truck    11
bus       4
Name: track_id, dtype: int64
